In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.


In [3]:
## defining constants used throughout the protocol

root3 = math.sqrt(3)
root2 = math.sqrt(2)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 


# This is the matrix to map states in the W basis to the standard basis,
# where the W basis is defined by:
# W = 1/root2(X + Z)
W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix) 

# This is the matrix to map states in the V basis to the standard basis,
# where the V basis is defined by:
# V = 1/root2(X - Z)
V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix) 

In [4]:
## Defines the auxilliary functions used throughout the protocol

def entangledPair():
    q = QuantumCircuit(2)
    q.h(1)
    q.cx(1,0)
    q.x(0)
    q.z(1)
    return q

# this creates a circuit that can be used to give a uniform random integer
# between 0 and 2 (inclusive). The circuit first applies a unitary
# operator that converts |0> -> (1/root3)|0> + (root2/root3)|1>. It then
# applies the Hadamard operator to the second qubit, giving 1/root2(|0> + |1>)
def random_3():
    q = QuantumCircuit(2)
    random_matrix = [[ 1 / root3, - root2 / root3 ],
                     [ root2 / root3, 1 / root3 ]]
    q.unitary(random_matrix, [1])
    q.h(0)
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return list(counts.keys())[0]

# This is a helper function that runs the circuit defined in random_3, to get a uniform
# random integer between 1 and 3. This random integer determines the random basis
# that Alice and Bob measure their qubit in

# Measuring the first qubit, there is a 1/3 chance that the first is 0, so if this occurs
# the result is 0. There is a 2/3 chance that the result is 1, and in this case
# we look to the second qubit. The result of measuring the second qubit is 0 or 1   
# with 1/2 chance each, meaning that in both qubits the measurements 10 and 11 each occur
# on average 1/3 of the time. In this case, the measurement result is converted to
# an integer (either 1 or 2), by summing the individual bits. 
# 1 is added to the result to keep the basis notation consistent with the assignment sheet
def measurement_choice():
    choice = random_3()
   
    if choice[0] == '0':
        choice = 0
    else:
        choice = int(choice[0]) + int(choice[1])
        
    return choice + 1


# This helper function rotates Alice's qubit from the basis it is being measured in,
# to the standard basis
def alice_basis_rotation(q, alice_basis):
    if alice_basis == 1:
        q.h(1)

    elif alice_basis == 2:
        q.unitary(W_transform,[1])

# This helper function rotates Bob's qubit from the basis it is being measured in,
# to the standard basis
def bob_basis_rotation(q, bob_basis):

    if bob_basis == 1:
        q.unitary(W_transform, [0])


    elif bob_basis == 3:
        q.unitary(V_transform, [0])



# A helper function for running a circuit once and returning the result
def simulate_circuit(q):
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    return list(counts.keys())[0]

In [5]:
## The main protocol

# N : average length of key produced

def ekert_91(N):
    alice_choices = []
    bob_choices = []

    alice_results = []
    bob_results = []

    # Receiving qubit, choosing basis randomly, and measuring in that basis
    for i in range ((9 * N) // 2):
        q = entangledPair()
    
        alice_choice = measurement_choice()
        alice_choices.append(alice_choice)
        
        bob_choice = measurement_choice()
        bob_choices.append(bob_choice)

        # converting Alice's qubit to her chosen basis
        alice_basis_rotation(q, alice_choice)

        # converting Bob's qubit to his chosen basis
        bob_basis_rotation(q, bob_choice)

        q.measure_all()

        result = simulate_circuit(q)

        ## Storing results
        
        alice_results.append(int(result[0]))

        # flipping Bob's bit
        bob_results.append(1 - int(result[1]))

    
    alice_key = []
    bob_key = []
    
    X_W = 0
    X_V = 0
    Z_W = 0
    Z_V = 0
    
    X_W_count = 0
    X_V_count = 0
    Z_W_count = 0
    Z_V_count = 0
    
    
    for i in range((9 * N) // 2):

        # Alice and Bob's choice of bases
        alice_choice = alice_choices[i]
        bob_choice = bob_choices[i]

        # Their results
        alice_bit = alice_results[i]
        bob_bit = bob_results[i]

        # if Alice and Bob are both measuring in the W basis
        if alice_choice == 2 and bob_choice == 1:
            alice_key.append(alice_bit)
            bob_key.append(bob_bit)

        # if Alice and Bob are both measuring in the Z basis
        elif alice_choices[i] == 3 and bob_choices[i] == 2:
            alice_key.append(alice_bit)
            bob_key.append(bob_bit)
            
        else:


            # converting {0,1} -> {+1,-1}
            alice_bit = 1 - 2 * alice_bit
            bob_bit = 1 - 2 * bob_bit
            
            product = alice_bit * bob_bit

            # Alice chooses X and Bob chooses W
            if alice_choices[i] == 1 and bob_choices[i] == 1:
                X_W += product
                X_W_count += 1

            # Alice chooses X and Bob chooses V
            elif alice_choices[i] == 1 and bob_choices[i] == 3:
                X_V += product
                X_V_count += 1

            # Alice chooses Z and Bob chooses W
            elif alice_choices[i] == 3 and bob_choices[i] == 1:
                Z_W += product
                Z_W_count += 1

            # Alice chooses Z and Bob chooses V
            elif alice_choices[i] == 3 and bob_choices[i] == 3:
                Z_V += product
                Z_V_count += 1
    
    X_W_average = X_W / X_W_count
    X_V_average = X_V / X_V_count
    Z_W_average = Z_W / Z_W_count
    Z_V_average = Z_V / Z_V_count
    
    s = abs(X_W_average - X_V_average + Z_W_average + Z_V_average)

    return {
        "alice" : alice_key,
        "bob" : bob_key,
        "s_value" : s
    }
    

In [6]:
# Since S is calculated from averages, the more measurements of <Z x W> etc we take, the more
# accurate S will be. This means that longer keys will produce, on average, a value of S that is
# closer to 2 * root2, since the process of producing a longer key means taking more
# measurements for the Bell Inequality.

results = ekert_91(100)
alice_key = results["alice"]
key_length = len(alice_key)
bob_key = results["bob"]
s = results["s_value"]
s_relative_error = abs(s - 2 * root2) / (2 * root2) * 100

print(f"Alice and Bob's keys are identical: {alice_key == bob_key}\n")
print(f"Final key: {alice_key} \n")
print(f"Key Length: {key_length}")
print(f"S Value: {s}")
print(f"Relative Error in S: {s_relative_error:.2f}%")

Alice and Bob's keys are identical: True

Final key: [0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0] 

Key Length: 103
S Value: 2.974474540821587
Relative Error in S: 5.16%
